# Hegselmann–Krause model: bounded confidence on a social network

Learning objective: see how averaging over compatible neighbors can produce agreement within opinion groups while disagreement persists across the population.

A social tie makes interaction possible, but an agent may listen only to neighbors whose opinions are sufficiently close. This lets us ask:

- Can a connected social network retain disagreement?
- How does the confidence bound change the outcome?
- What changes when influence weights depend on current opinions?

The model makes these assumptions explicit so that we can examine their consequences.

## The Hegselmann–Krause model on networks

Each agent $i$ has a continuous opinion $x_i(t)\in[0,1]$. Let $\mathcal{N}_i$ be its neighbors in the fixed social network. Its confidence neighborhood at round $t$ is

$$
\mathcal{C}_i(t)=\{i\}\cup
\{j\in\mathcal{N}_i:\ |x_j(t)-x_i(t)|\leq\epsilon\}.
$$

Every agent updates synchronously by taking an equal-weight average:

$$
x_i(t+1)=\frac{1}{|\mathcal{C}_i(t)|}
\sum_{j\in\mathcal{C}_i(t)}x_j(t).
$$

All confidence tests and averages use the unchanged opinions from round $t$. Including the agent itself ensures that the average is always defined.

### Conventions used here

- **Opinion space:** continuous values in $[0,1]$.
- **Population:** 100 agents by default, controlled by N_AGENTS.
- **Topology:** one fixed, simple, connected, undirected Watts–Strogatz network, using the DeGroot notebook's network settings.
- **Initial state:** independent uniform draws on $[0,1]$ with OPINION_SEED.
- **Influence:** uniform over the agent itself and its currently compatible social neighbors.
- **Confidence boundary:** inclusive inequality $|x_i-x_j|\leq\epsilon$; pairs exactly on the boundary are included.
- **Update schedule:** synchronous, deterministic once the network and initial opinions are fixed; no pair sampling.
- **Time:** one round is one complete population update.
- **Simulation horizon:** exactly N_ROUNDS; include round zero and do not stop early.
- **Isolated agents:** retain their opinions because they average only themselves.
- **Observables:** opinion trajectories, global opinion range, population mean, and final opinion distribution.
- **Consensus criterion:** global opinion range strictly below CONSENSUS_TOLERANCE.
- **Parameter ranges:** integer N_AGENTS >= 3 and N_ROUNDS >= 0; integer $1\leq$ NEIGHBORS_ON_EACH_SIDE $<N/2$; rewiring probability and confidence bounds in $[0,1]$; CONSENSUS_TOLERANCE > 0.

This is a network-restricted version of HK. A complete social graph recovers the usual homogeneous-confidence model in which all agents can consider one another.

The influence matrix changes with opinions. Its rows sum to one, but its columns generally do not: the arithmetic mean is not generally conserved. Connectedness of the social network does not by itself ensure consensus under bounded confidence.

In [ ]:
import random

import igraph as ig
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import numpy as np

# Adjustable parameters
N_AGENTS = 100
N_ROUNDS = 100
NEIGHBORS_ON_EACH_SIDE = 3
REWIRING_PROBABILITY = 0.10
CONFIDENCE_BOUND = 0.20
COMPARISON_CONFIDENCE_BOUND = 1.00
CONSENSUS_TOLERANCE = 0.01

# Reuse DeGroot's network and initial opinions for a familiar setting.
NETWORK_SEED = 2026
OPINION_SEED = 2027
LAYOUT_SEED = 2028

assert isinstance(N_AGENTS, int) and N_AGENTS >= 3
assert isinstance(N_ROUNDS, int) and N_ROUNDS >= 0
assert isinstance(NEIGHBORS_ON_EACH_SIDE, int)
assert 1 <= NEIGHBORS_ON_EACH_SIDE < N_AGENTS / 2
assert 0 <= REWIRING_PROBABILITY <= 1
assert 0 <= CONFIDENCE_BOUND <= 1
assert 0 <= COMPARISON_CONFIDENCE_BOUND <= 1
assert CONSENSUS_TOLERANCE > 0

plt.rcParams.update({
    "figure.figsize": (9, 5),
    "font.size": 12,
    "axes.titlesize": 14,
    "axes.labelsize": 12,
})

## 1. Create the social network

We use `igraph` to create, store, lay out, and draw a small-world network. The parameter `NEIGHBORS_ON_EACH_SIDE = 3` starts each agent with three lattice neighbors on either side.

The `python-igraph` generator uses a small-world rewiring variant. This synthetic topology is a convenient theoretical influence structure, not a reconstruction of a real social network.

The force-directed layout affects only the drawing, not the influence matrix or dynamics. Fixed seeds make the generated network and layout reproducible within the specified software environment; results may differ across library versions.

In [ ]:
# igraph uses this seeded generator for reproducible rewiring.
ig.set_random_number_generator(random.Random(NETWORK_SEED))

graph_ig = ig.Graph.Watts_Strogatz(
    dim=1,
    size=N_AGENTS,
    nei=NEIGHBORS_ON_EACH_SIDE,
    p=REWIRING_PROBABILITY,
)

assert not graph_ig.is_directed()
assert graph_ig.vcount() == N_AGENTS
assert graph_ig.is_simple(), "The influence network must be simple."
assert graph_ig.is_connected(), "The influence network must be connected."
assert graph_ig.vs.indices == list(range(N_AGENTS))

# Seed the starting coordinates independently from the network draw.
ig.set_random_number_generator(random.Random(LAYOUT_SEED))
layout_rng = np.random.default_rng(LAYOUT_SEED)
layout_start = layout_rng.uniform(-1.0, 1.0, size=(N_AGENTS, 2))
layout = graph_ig.layout_fruchterman_reingold(
    seed=layout_start.tolist(),
)
assert len(layout) == N_AGENTS

opinion_rng = np.random.default_rng(OPINION_SEED)
initial_opinions = opinion_rng.uniform(0.0, 1.0, size=N_AGENTS)

print(f"Agents: {graph_ig.vcount()}")
print(f"Connections: {graph_ig.ecount()}")
print(f"Connected: {graph_ig.is_connected()}")

In [ ]:
opinion_norm = mcolors.Normalize(vmin=0.0, vmax=1.0)
opinion_cmap = plt.colormaps["viridis"]
opinion_color_scale = plt.cm.ScalarMappable(
    norm=opinion_norm,
    cmap=opinion_cmap,
)


def draw_opinion_network(ax, opinions, title, vertex_size):
    """Draw opinions on the igraph network using vertex-ID order."""
    vertex_colors = [
        mcolors.to_hex(opinion_cmap(opinion_norm(value)))
        for value in opinions
    ]
    assert len(vertex_colors) == graph_ig.vcount()

    ig.plot(
        graph_ig,
        target=ax,
        layout=layout,
        vertex_color=vertex_colors,
        vertex_size=vertex_size,
        vertex_frame_color="white",
        vertex_frame_width=0.4,
        vertex_label=None,
        edge_color="#b0b4b8",
        edge_width=0.7,
        autocurve=False,
    )
    ax.set_title(title)
    ax.axis("off")


fig, ax = plt.subplots(figsize=(9, 7), layout="constrained")
draw_opinion_network(
    ax,
    initial_opinions,
    "Initial opinions on the network",
    vertex_size=12,
)
fig.colorbar(opinion_color_scale, ax=ax, label="Initial opinion")
plt.show()

## 2. Translate the theory into code

The social adjacency matrix identifies possible interactions. At each round, we retain only ties between agents whose current opinion difference is at most the confidence bound. We then include self-influence and normalize every row.

With $W_{ij}(t)$ denoting agent $j$'s influence on agent $i$,

$$
W_{ij}(t)=
\begin{cases}
1/|\mathcal{C}_i(t)|,&j\in\mathcal{C}_i(t),\\
0,&\text{otherwise},
\end{cases}
\qquad
x(t+1)=W(t)x(t).
$$

The social graph stays fixed; the set of compatible ties is recomputed and can change. A compatible neighbor is counted once, and self-influence is counted once. With a confidence bound of $1$, every social neighbor is eligible for opinions in $[0,1]$, yielding the fixed equal-neighbor DeGroot matrix.

In [ ]:
adjacency = np.asarray(graph_ig.get_adjacency().data, dtype=float)
assert adjacency.shape == (N_AGENTS, N_AGENTS)
assert np.array_equal(adjacency, adjacency.T)
assert np.all(np.diag(adjacency) == 0)
assert np.all((adjacency == 0) | (adjacency == 1))


def confidence_weights(adjacency, opinions, confidence_bound):
    """Build row-stochastic influence weights from one frozen opinion state."""
    opinion_distances = np.abs(opinions[:, None] - opinions[None, :])
    compatible_connections = (
        (adjacency > 0) & (opinion_distances <= confidence_bound)
    )
    np.fill_diagonal(compatible_connections, True)
    row_totals = compatible_connections.sum(axis=1, keepdims=True)
    return compatible_connections / row_totals


initial_weights = confidence_weights(adjacency, initial_opinions, CONFIDENCE_BOUND)
assert np.all(initial_weights >= 0)
assert np.allclose(initial_weights.sum(axis=1), 1.0)
assert np.all(np.diag(initial_weights) > 0)

print("First five row sums:", initial_weights.sum(axis=1)[:5])
print(
    "Mean initial confidence-neighborhood size (including self):",
    f"{np.count_nonzero(initial_weights, axis=1).mean():.2f}",
)

## 3. Run synchronous Hegselmann–Krause updates

The function stores the initial opinions and every later round. Each round rebuilds the confidence weights from a frozen copy of the previous state, then updates all agents together.

A three-agent path provides a hand-check of synchrony and shows why the ordinary mean need not be preserved. All three agents must use the old middle opinion, even though that agent also updates.

In [ ]:
def simulate_hk(adjacency, initial_state, n_rounds, confidence_bound):
    """Return opinions from round 0 through n_rounds under network HK."""
    opinions = np.asarray(initial_state, dtype=float)
    adjacency = np.asarray(adjacency, dtype=float)
    assert opinions.ndim == 1 and opinions.size > 0
    assert np.all(np.isfinite(opinions)) and np.all((0 <= opinions) & (opinions <= 1))
    assert adjacency.shape == (len(opinions), len(opinions))
    assert np.all((adjacency == 0) | (adjacency == 1))
    assert np.array_equal(adjacency, adjacency.T)
    assert np.all(np.diag(adjacency) == 0)
    assert isinstance(n_rounds, int) and n_rounds >= 0
    assert 0 <= confidence_bound <= 1

    history = np.empty((n_rounds + 1, len(opinions)))
    history[0] = opinions

    for round_index in range(n_rounds):
        previous_opinions = history[round_index].copy()
        weights = confidence_weights(adjacency, previous_opinions, confidence_bound)
        history[round_index + 1] = weights @ previous_opinions

    return history


# A path 0--1--2: all social edges are compatible, including one at the boundary.
test_adjacency = np.array([[0, 1, 0], [1, 0, 1], [0, 1, 0]])
test_initial = np.array([0.0, 0.5, 0.75])
test_history = simulate_hk(test_adjacency, test_initial, 1, confidence_bound=0.5)
assert np.allclose(test_history[1], [0.25, 5 / 12, 0.625])
assert not np.isclose(test_history[1].mean(), test_initial.mean())


def check_averaging_invariants(history, initial_state):
    """Check finite values, the convex hull, and nonincreasing opinion range."""
    assert np.all(np.isfinite(history))
    assert history.min() >= initial_state.min() - 1e-12
    assert history.max() <= initial_state.max() + 1e-12
    assert np.all(np.diff(history.min(axis=1)) >= -1e-12)
    assert np.all(np.diff(history.max(axis=1)) <= 1e-12)


opinion_history = simulate_hk(
    adjacency, initial_opinions, N_ROUNDS, CONFIDENCE_BOUND,
)
check_averaging_invariants(opinion_history, initial_opinions)
final_opinions = opinion_history[-1]

## 4. Compare the initial and final states

Both panels use the same network layout and color scale. This makes changes in opinions visible without confusing them with changes in node position.

In [ ]:
fig, axes = plt.subplots(
    1,
    2,
    figsize=(14, 6),
    layout="constrained",
)

states = [
    (initial_opinions, "Initial opinions"),
    (final_opinions, f"Opinions after {N_ROUNDS} rounds"),
]

for ax, (opinions, title) in zip(axes, states):
    draw_opinion_network(
        ax,
        opinions,
        title,
        vertex_size=10,
    )

fig.colorbar(
    opinion_color_scale,
    ax=axes,
    label="Opinion",
    shrink=0.75,
)
fig.suptitle(
    "The same network before and after Hegselmann–Krause"
)
plt.show()

## 5. A typical analysis: does disagreement disappear?

We follow opinion trajectories and the global opinion range:

$$
R(t)=\max_i x_i(t)-\min_i x_i(t).
$$

Convex averaging keeps every new opinion inside the previous population's range, so this range cannot increase. It may nevertheless remain positive when confidence prevents influence between groups.

The dashed line in the trajectory plot is the evolving population mean. It is an observable, not a conserved quantity or a predicted consensus value. Small changes at the end of a finite run do not prove asymptotic convergence or establish an exact cluster count.

In [ ]:
rounds = np.arange(N_ROUNDS + 1)
opinion_range = np.ptp(opinion_history, axis=1)
population_mean = opinion_history.mean(axis=1)
consensus_rounds = np.flatnonzero(opinion_range < CONSENSUS_TOLERANCE)
first_consensus_round = int(consensus_rounds[0]) if consensus_rounds.size else None

fig, axes = plt.subplots(1, 2, figsize=(14, 5), layout="constrained")
axes[0].plot(rounds, opinion_history, color="#377eb8", alpha=0.30, linewidth=1)
axes[0].plot(
    rounds, population_mean, color="#d95f02", linestyle="--",
    linewidth=2, label="Population mean",
)
axes[0].set(
    title="Individual opinion trajectories", xlabel="Round", ylabel="Opinion",
    ylim=(0, 1), xlim=(0, max(1, N_ROUNDS)),
)
axes[0].legend(frameon=True, facecolor="white", edgecolor="none", framealpha=1)

axes[1].plot(
    rounds, opinion_range, color="#d95f02", linewidth=2.5, label="Opinion range",
)
axes[1].axhline(
    CONSENSUS_TOLERANCE, color="#333333", linestyle="--",
    label=f"Consensus tolerance = {CONSENSUS_TOLERANCE}",
)
if first_consensus_round is not None:
    axes[1].axvline(
        first_consensus_round, color="#377eb8", linestyle=":",
        label=f"First detected: round {first_consensus_round}",
    )
axes[1].set(
    title="Disagreement over time", xlabel="Round",
    ylabel="Maximum opinion - minimum opinion",
    ylim=(0, 1), xlim=(0, max(1, N_ROUNDS)),
)
axes[1].legend(frameon=False)
fig.suptitle(f"Network HK dynamics: confidence bound = {CONFIDENCE_BOUND:g}")
plt.show()

if first_consensus_round is None:
    print("The trajectory did not reach the consensus tolerance.")
else:
    print("First round below the consensus tolerance:", first_consensus_round)
print(f"Initial mean: {population_mean[0]:.6f}")
print(f"Final mean: {population_mean[-1]:.6f}")
print(f"Final opinion range: {opinion_range[-1]:.6f}")
if N_ROUNDS > 0:
    print(
        "Largest opinion change in the last round:",
        f"{np.max(np.abs(opinion_history[-1] - opinion_history[-2])):.6g}",
    )

## 6. One focused experiment: change the confidence bound

Keep the social graph and initial opinions fixed, and change only the confidence bound. No interaction seed is needed: synchronous HK is deterministic once those inputs are fixed.

The default comparison uses a bound of $1$, making every social neighbor eligible throughout the run. This exactly recovers the equal-neighbor DeGroot dynamics from the first notebook. A numerical check below verifies this connection.

The histograms use shared bins and axes. Their peaks show opinion concentrations at the chosen bin width; they are not an exact count of limiting clusters. Two finite runs do not establish a universal confidence threshold for consensus.

In [ ]:
comparison_history = simulate_hk(
    adjacency, initial_opinions, N_ROUNDS, COMPARISON_CONFIDENCE_BOUND,
)
check_averaging_invariants(comparison_history, initial_opinions)

# At confidence bound 1, HK must reproduce fixed-matrix DeGroot.
degroot_connections = adjacency + np.eye(N_AGENTS)
degroot_weights = degroot_connections / degroot_connections.sum(axis=1, keepdims=True)
unrestricted_history = simulate_hk(adjacency, initial_opinions, N_ROUNDS, 1.0)
assert np.allclose(
    unrestricted_history[1:], unrestricted_history[:-1] @ degroot_weights.T,
    atol=1e-12, rtol=0,
)

fig, axes = plt.subplots(
    1, 2, figsize=(14, 5), sharex=True, sharey=True, layout="constrained",
)
scenarios = [
    (CONFIDENCE_BOUND, opinion_history, "#377eb8"),
    (COMPARISON_CONFIDENCE_BOUND, comparison_history, "#d95f02"),
]
bins = np.linspace(0, 1, 21)
for ax, (confidence_bound, history, color) in zip(axes, scenarios):
    ax.hist(history[-1], bins=bins, color=color, edgecolor="white")
    ax.axvline(
        initial_opinions.mean(), color="#333333", linestyle="--",
        label="Initial mean (reference)",
    )
    ax.set(
        title=f"Confidence bound = {confidence_bound:g}",
        xlabel="Opinion", ylabel="Number of agents",
        xlim=(0, 1), ylim=(0, N_AGENTS),
    )
    ax.legend(frameon=False)
    final_range = np.ptp(history[-1])
    print(
        f"Confidence {confidence_bound:g}: final range = {final_range:.6f}; "
        f"consensus detected = {final_range < CONSENSUS_TOLERANCE}; "
        f"final mean = {history[-1].mean():.6f}"
    )
fig.suptitle(f"Final opinions after {N_ROUNDS} rounds")
plt.show()

## Interpretation and limitations

HK combines synchronous averaging with opinion-dependent confidence neighborhoods. The finite trajectories illustrate how bounded confidence can sustain distinct opinion concentrations on a connected social graph.

At confidence bound $1$, this network version recovers DeGroot with fixed equal-neighbor weights. With smaller bounds, compatible ties can appear or disappear, changing the weights over time. The arithmetic mean generally changes; the degree-weighted consensus formula from the fixed-network DeGroot example does not generally apply to this changing confidence graph.

The social network remains fixed throughout. A disconnected confidence graph at one round is not by itself proof of permanent fragmentation: later opinions can change which ties are compatible. Global range and finite-time histograms describe observations, not a theorem about the final number of clusters.

The model assumes a common confidence bound, equal influence within each confidence neighborhood, scalar opinions, and simultaneous updating. It omits external information, changing social ties, and individual differences in susceptibility. Its outcomes are consequences of these assumptions, not predictions of real societies.

### Suggested experiments

1. Change `CONFIDENCE_BOUND` while keeping the network and initial-opinion seeds fixed.
2. Increase `N_ROUNDS` to distinguish slow relaxation from apparent finite-time plateaus.
3. Change `NEIGHBORS_ON_EACH_SIDE` or `REWIRING_PROBABILITY` and repeat across several network and opinion seeds.

### Original reference

Hegselmann, R., & Krause, U. (2002). *Opinion dynamics and bounded confidence: models, analysis and simulation*. Journal of Artificial Societies and Social Simulation, 5(3), article 2. [Original article](https://www.jasss.org/5/3/2.html).

This notebook adds the restriction to neighbors of a fixed social graph; a complete graph gives the usual homogeneous-confidence HK model.